# 02 — Data Validation

**Purpose:** Verify that the three raw datasets meet minimum quality 
standards before any modeling touches them. A model trained on silent 
data corruption produces confident but wrong results.

**Validation module:** `src/data/validate.py`  

**Checks per dataset:** row count, null values, plausible value ranges.

**Outline:**

0. Setup
1. Validation Results
2. Demonstrating the Validator Catches Problems
3. Conclusion

---
---
## 0. Imports and Path Setup

All imports live in one cell at the top. `ROOT` points to the project root so every subsequent path is absolute and works regardless of where the notebook is opened from.

In [5]:
import sys
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.data.validate import run_all        # ← ImportError: attempted relative…

mxn = pd.read_csv(ROOT / "data/raw/mxn_usd.csv", index_col="Date", parse_dates=True)
ipc = pd.read_csv(ROOT / "data/raw/ipc.csv", index_col="Date", parse_dates=True)
macro = pd.read_csv(ROOT / "data/raw/macro.csv", index_col=0, parse_dates=True)

print(mxn.head())
print(ipc.head())
print(macro.head())

            MXN_USD
Date               
2000-01-03   9.3949
2000-01-04   9.4564
2000-01-05   9.5455
2000-01-06   9.5705
2000-01-07   9.5150
                    IPC
Date                   
2001-03-06  6252.950195
2001-03-07  6331.319824
2001-03-08  6361.069824
2001-03-09  6270.959961
2001-03-12  5963.370117
            VIXCLS   DFF  T10Y2Y
2000-01-01     NaN  3.99     NaN
2000-01-02     NaN  3.99     NaN
2000-01-03   24.21  5.43    0.20
2000-01-04   27.01  5.38    0.19
2000-01-05   26.41  5.41    0.24


---
---
## 1. Validation Results
All three datasets are validated in a single call. `raise_on_failure=False` 
means failures are reported, not raised — appropriate for a notebook where 
we want to inspect results interactively. In the production pipeline, 
`raise_on_failure=True` hard-stops execution if data is bad.

In [8]:
results = run_all(mxn, ipc, macro, raise_on_failure=False)
print("============= Global results ============= \n")
print(results)
print("\n\n")
print("============= IPC results ============= \n")
print(results["ipc"])

============= Global results ============= 

{'mxn': {'passed': True, 'failures': []}, 'ipc': {'passed': True, 'failures': []}, 'macro': {'passed': True, 'failures': []}, 'all_passed': True}



============= IPC results ============= 

{'passed': True, 'failures': []}


---
---
## 2. Demonstrating the Validator Catches Real Problems

During development, IPC had 60 null values caused by calendar misalignment 
between the Mexican market and the Banxico series. Below we reproduce that 
failure by injecting nulls, then show the clean data passing.

In [9]:
# Reproduce the original failure
ipc_with_nulls = ipc.copy()
ipc_with_nulls.iloc[10:70] = None

demo = run_all(mxn, ipc_with_nulls, macro, raise_on_failure=False)
print("With nulls:  ", demo["ipc"])

real = run_all(mxn, ipc, macro, raise_on_failure=False)
print("Clean data:  ", real["ipc"])

With nulls:   {'passed': False, 'failures': ["60 null values found in 'IPC'."]}
Clean data:   {'passed': True, 'failures': []}


---
---
## 3. Conclusion

All three datasets pass validation. The data is clean and ready for 
exploratory analysis in `03_eda.ipynb`.

| Dataset | Rows | Nulls | Range check | Passed |
|---------|------|-------|-------------|--------|
| MXN/USD | ✅   | ✅    | [3, 30]  ✅  | ✅     |
| IPC     | ✅   | ✅    | > 0  ✅      | ✅     |
| Macro   | ✅   | —     | VIX [0,90] ✅| ✅     |